# INSTRUCTOR GUIDE: Week 7 Session 2 Decision Tree + evaluation metrics
Not for students. Solutions are filled in below each Your turn. Keep `greenline_deliveries.csv` in the same folder as this notebook. This notebook repeats the prep from Session 1 so it stands alone.

## Run of show (2 hours)
| Time | Block | What happens |
|---|---|---|
| 0:00 to 0:10 | Check-in | Who has started the ML project? Which dataset? Any capstone notes? |
| 0:10 to 0:30 | Concept: what a tree does, and how to read the metrics | Draw a tiny tree. MAE / RMSE / R-squared for numbers; accuracy, precision, recall, confusion matrix for classes. The 75 percent dummy is the running example. |
| 0:30 to 0:55 | Live coding: prep (quick), logistic recap, Decision Tree classifier | Depth 4 vs logistic. Read the matrix in sentences. Your turn 1 and 2. |
| 0:55 to 1:05 | Break | |
| 1:05 to 1:35 | Regression tree vs linear, then overfitting | Depth 4 tree vs linear MAE. Then depth 12: tiny training error, worse test error. Your turn 3. |
| 1:35 to 1:50 | Challenge | Fit depth 12 for classification; explain the train/test gap. |
| 1:50 to 2:00 | Recap and project close-out | Re-state the brief. No new methods after this week for the capstone unless they already have them. |

**If time runs short:** keep the confusion-matrix reading and the overfitting demo. The regression tree can be a shorter comparison.

## Talking points
- A decision tree asks a sequence of yes/no questions about the features (is distance greater than 15 km? is it the weekend?) and lands in a leaf with a prediction. `max_depth` caps how many questions it may stack.
- More depth is not automatically better. A deep tree can memorise the training rows (overfitting) and then do worse on new rows. We always report train *and* test scores when we change depth.
- Logistic regression beat the depth-4 tree on this test set (accuracy 0.88 vs 0.82). That is a useful surprise: a simpler model can win. The project should compare, not assume the tree is the upgrade.
- Accuracy of 0.75 for the dummy is the object lesson. Precision and recall, plus the matrix, are how we tell the story.
- For times: linear regression MAE about 3.9 vs a depth-4 tree MAE about 6.2. Linear is better here. The depth-12 tree gets training MAE close to 0 and a worse test MAE — a clean overfitting picture.
- Interpret every metric in a sentence. A number without a sentence is not finished.

## Common mistakes to watch for
- Accuracy-only on the imbalanced late/on-time split.
- Ignoring the confusion matrix, or reading it the wrong way round.
- Metrics with no interpretation.
- Using `delivery_minutes` as a feature for `arrived_late`.
- Reporting only the training score after raising `max_depth`.
- No baseline in the project notebook.
- Assuming a tree must beat logistic or linear regression.

## Expected numbers (same split as Session 1)
- Dummy classifier: accuracy 0.75, precision 0, recall 0, matrix [[75, 0], [25, 0]].
- Logistic: accuracy 0.88, precision 0.741, recall 0.80, matrix [[68, 7], [5, 20]].
- Tree depth 4: train accuracy 0.917, test accuracy 0.82, precision 0.621, recall 0.72, matrix [[64, 11], [7, 18]].
- Tree depth 3 (Your turn): train 0.903, test 0.82, precision 0.613, recall 0.76, matrix [[63, 12], [6, 19]].
- Tree depth 12: train accuracy 1.0, test accuracy 0.81, precision 0.615, recall 0.64, matrix [[65, 10], [9, 16]].
- Linear regression: test MAE 3.87, RMSE 4.76, R-squared 0.938.
- Tree regressor depth 4: test MAE 6.16, RMSE 7.86, R-squared 0.83.
- Tree regressor depth 12: train MAE 0.06, test MAE 6.89.


# Week 7 Session 2: Decision Trees and honest evaluation
**Forge Academy | Data Science Foundations**

*Covers Week 7 Session 2: Decision Trees, and reading MAE, RMSE, R-squared, accuracy, precision, recall and the confusion matrix.*

## Mission

GreenLine still wants two honest answers: how long will a drop take, and will it be late? Last session we fitted Linear and Logistic Regression and beat two simple baselines. Today we add a Decision Tree, we compare it with those models, and we practise saying what the metrics mean until the sentences feel ordinary.

The comparison is the point. A tree is not automatically an upgrade.

Keep `greenline_deliveries.csv` in the same folder as this notebook and run the cells from top to bottom. The preparation below repeats Session 1 so this notebook stands on its own.


## 0. Metrics, in plain words


When the target is a **number** (delivery time):

- **MAE** — the average size of the mistake, in the same units as the target (minutes). Easy to say out loud.
- **RMSE** — also in minutes, but large mistakes count more.
- **R-squared** — how much of the variation in the target the model explains. 1 is a perfect fit on that set; a mean baseline sits near 0.

When the target is a **class** (late or on time):

- **Accuracy** — the share of rows we got right. Misleading if one class is much more common.
- **Precision** — of the rows we *called* late, how many really were late?
- **Recall** — of the rows that *really were* late, how many did we catch?
- **Confusion matrix** — the four counts: correctly on time, false late, missed late, correctly late.

A Decision Tree asks a sequence of yes or no questions about the features, and lands in a leaf with a prediction. `max_depth` is the cap on how many questions it may stack. A short tree is often a little rough; a very deep tree can memorise the training rows and then stumble on new ones. That problem is called **overfitting**, and we will look at it with numbers, not just the word.


## 1. Prepare the table (same steps as Session 1)


We will use the same preparation in both Week 7 sessions, so this notebook stands on its own.

The habit is: work on a copy, fill the small number of missing package weights with the median, and turn `area` into 0/1 columns. We drop `area_Mainland` and treat Mainland as the reference area. Coefficients for Island and Outskirts will then mean "compared with Mainland".

Do not put `delivery_minutes` into the feature list when the target is `arrived_late`. Late deliveries are the long ones, so that would leak the answer.


In [ ]:
import pandas as pd


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression


In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix


In [ ]:
raw = pd.read_csv("greenline_deliveries.csv")


In [ ]:
work = raw.copy()


In [ ]:
package_median = work["package_kg"].median()


In [ ]:
work["package_kg"] = work["package_kg"].fillna(package_median)


In [ ]:
area_flags = pd.get_dummies(work["area"], prefix="area")


In [ ]:
area_flags = area_flags.drop(columns=["area_Mainland"])


In [ ]:
work = pd.concat([work, area_flags], axis=1)


In [ ]:
feature_names = [
    "distance_km",
    "package_kg",
    "is_weekend",
    "rain_mm",
    "depart_hour",
    "rider_experience_months",
    "area_Island",
    "area_Outskirts",
]


In [ ]:
X = work[feature_names]


In [ ]:
y_time = work["delivery_minutes"]


In [ ]:
y_late = work["arrived_late"]


In [ ]:
X_train, X_test, y_time_train, y_time_test = train_test_split(X, y_time, test_size=0.25, random_state=42)


In [ ]:
X_train_c, X_test_c, y_late_train, y_late_test = train_test_split(X, y_late, test_size=0.25, random_state=42, stratify=y_late)


In [ ]:
print(len(X_train), len(X_test), len(X_train_c), len(X_test_c))


## 2. Recap the two baselines and Logistic Regression


In [ ]:
late_baseline = DummyClassifier(strategy="most_frequent")


In [ ]:
late_baseline.fit(X_train_c, y_late_train)


In [ ]:
y_late_baseline = late_baseline.predict(X_test_c)


In [ ]:
print(accuracy_score(y_late_test, y_late_baseline))


In [ ]:
print(precision_score(y_late_test, y_late_baseline, zero_division=0))


In [ ]:
print(recall_score(y_late_test, y_late_baseline, zero_division=0))


In [ ]:
print(confusion_matrix(y_late_test, y_late_baseline))


In [ ]:
late_model = LogisticRegression(max_iter=1000, random_state=42)


In [ ]:
late_model.fit(X_train_c, y_late_train)


In [ ]:
y_late_log = late_model.predict(X_test_c)


In [ ]:
print(accuracy_score(y_late_test, y_late_log))


In [ ]:
print(precision_score(y_late_test, y_late_log, zero_division=0))


In [ ]:
print(recall_score(y_late_test, y_late_log, zero_division=0))


In [ ]:
print(confusion_matrix(y_late_test, y_late_log))


The dummy still looks "75 percent accurate" and still catches no late drops. Logistic regression should again show accuracy around 0.88, precision around 0.74, recall around 0.80, and a matrix like `[[68, 7], [5, 20]]`.


**INSTRUCTOR NOTE.** Same numbers as Session 1. If anyone's matrix differs, they probably dropped a different area column, filled missing values differently, or changed `random_state`. Rebuild from the top.


## 3. A Decision Tree for late deliveries

We start with `max_depth=4`, a short tree that is easy to talk about. We then look at training accuracy and test accuracy together.


In [ ]:
late_tree = DecisionTreeClassifier(max_depth=4, random_state=42)


In [ ]:
late_tree.fit(X_train_c, y_late_train)


In [ ]:
y_late_tree = late_tree.predict(X_test_c)


In [ ]:
print(accuracy_score(y_late_train, late_tree.predict(X_train_c)))


In [ ]:
print(accuracy_score(y_late_test, y_late_tree))


In [ ]:
print(precision_score(y_late_test, y_late_tree, zero_division=0))


In [ ]:
print(recall_score(y_late_test, y_late_tree, zero_division=0))


In [ ]:
print(confusion_matrix(y_late_test, y_late_tree))


A picture of the tree helps, as long as we keep it shallow enough to read.


In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(late_tree, feature_names=feature_names, class_names=["on time", "late"], filled=True, max_depth=3, fontsize=8)
plt.title("Decision tree for late deliveries (first layers)")
plt.show()


On this test set the depth-4 tree is usually *behind* logistic regression (accuracy around 0.82 against 0.88, recall around 0.72 against 0.80). That is worth sitting with: a newer-looking model is not automatically better. In the project, you will compare and you will say which one you would actually use, and why.


**INSTRUCTOR NOTE.** Depth 4: train accuracy 0.917, test 0.82, precision 0.621, recall 0.72, matrix [[64, 11], [7, 18]]. Compared with logistic (0.88 / 0.741 / 0.80, matrix [[68, 7], [5, 20]]), the tree has more false alarms (11 vs 7) and more misses (7 vs 5). Do not skip this comparison.


### Your turn 1

Fit a `DecisionTreeClassifier` with `max_depth=3` and `random_state=42`, on the same training data. Print training accuracy, test accuracy, test precision, test recall, and the confusion matrix.


In [ ]:
# SOLUTION
tree_d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_d3.fit(X_train_c, y_late_train)
print(accuracy_score(y_late_train, tree_d3.predict(X_train_c)))
print(accuracy_score(y_late_test, tree_d3.predict(X_test_c)))
print(precision_score(y_late_test, tree_d3.predict(X_test_c), zero_division=0))
print(recall_score(y_late_test, tree_d3.predict(X_test_c), zero_division=0))
print(confusion_matrix(y_late_test, tree_d3.predict(X_test_c)))


**INSTRUCTOR NOTE. Your turn 1.** Train 0.903, test 0.82, precision 0.613, recall 0.76, matrix [[63, 12], [6, 19]]. Test accuracy matches depth 4, recall is a little higher (19 caught vs 18), precision a little lower. A good moment to say: changing one number (`max_depth`) moves the four counts, so we have to read all of them.


### Your turn 2

Write three short sentences, using the numbers already on the screen:

1. What does the dummy's accuracy of 0.75 fail to tell us?
2. Which of logistic regression and the depth-4 tree would you prefer for catching late drops, and which number are you using to decide?
3. In the depth-4 confusion matrix, what do the 11 and the 7 mean?


**SOLUTION — Your turn 2.**

1. It fails to tell us that the dummy never identifies a late drop (precision and recall are both 0).
2. Logistic regression, because its recall is about 0.80 against about 0.72 for the depth-4 tree (it also has fewer false alarms). Students may reasonably watch precision instead; ask them to name the number they used.
3. 11 is false late alarms (on-time drops called late). 7 is missed late drops (late drops called on time).

**INSTRUCTOR NOTE.** If the wording is reversed, redraw the matrix. Do not let "11 and 7" stay vague.


## 4. A Decision Tree for delivery time

Same idea, different target. We compare a depth-4 tree with last session's Linear Regression, using MAE, RMSE and R-squared on the test set.


In [ ]:
time_baseline = DummyRegressor(strategy="mean")


In [ ]:
time_baseline.fit(X_train, y_time_train)


In [ ]:
y_time_baseline = time_baseline.predict(X_test)


In [ ]:
time_linear = LinearRegression()


In [ ]:
time_linear.fit(X_train, y_time_train)


In [ ]:
y_time_linear = time_linear.predict(X_test)


In [ ]:
time_tree = DecisionTreeRegressor(max_depth=4, random_state=42)


In [ ]:
time_tree.fit(X_train, y_time_train)


In [ ]:
y_time_tree = time_tree.predict(X_test)


In [ ]:
print(mean_absolute_error(y_time_test, y_time_baseline))


In [ ]:
print(mean_absolute_error(y_time_test, y_time_linear))


In [ ]:
print(mean_absolute_error(y_time_test, y_time_tree))


In [ ]:
print(mean_squared_error(y_time_test, y_time_linear) ** 0.5)


In [ ]:
print(mean_squared_error(y_time_test, y_time_tree) ** 0.5)


In [ ]:
print(r2_score(y_time_test, y_time_linear))


In [ ]:
print(r2_score(y_time_test, y_time_tree))


Both models should beat the mean baseline (MAE about 16.8). Linear regression should still be ahead of the depth-4 tree here (MAE about 3.9 against about 6.2; R-squared about 0.94 against about 0.83). Again: compare, then choose, and say why.


**INSTRUCTOR NOTE.** Baseline MAE 16.80. Linear MAE 3.87, RMSE 4.76, R-squared 0.938. Tree depth 4 MAE 6.16, RMSE 7.86, R-squared 0.83. Linear wins cleanly on this data because the true relationship is close to a weighted sum, which is exactly what linear regression assumes.


### Your turn 3

Print the test MAE for the baseline, the linear model, and the depth-4 tree, then write two sentences saying which model you would use to promise a delivery time, and what a MAE of about 4 minutes means for a customer.


In [ ]:
# SOLUTION
print(mean_absolute_error(y_time_test, y_time_baseline))
print(mean_absolute_error(y_time_test, y_time_linear))
print(mean_absolute_error(y_time_test, y_time_tree))


**SOLUTION — interpretation.** The linear model is the one to use on this data: its average mistake is about 4 minutes, against about 6 minutes for the tree and about 17 minutes for the mean baseline. A MAE of about 4 minutes means that, on typical test drops, the promised time is off by roughly four minutes (some more, some less), which is tight enough to be useful and still not a guarantee.

**INSTRUCTOR NOTE. Your turn 3.** If someone prefers the tree "because it is more advanced", push gently back with the MAE numbers. That is the whole week's point.


## 5. What happens if the tree is allowed to grow too deep?

We now fit a much deeper tree (`max_depth=12`) for both problems, and we look at training scores and test scores together. If training performance looks almost perfect while test performance gets worse, the tree has started to memorise the training rows.


In [ ]:
deep_late = DecisionTreeClassifier(max_depth=12, random_state=42)


In [ ]:
deep_late.fit(X_train_c, y_late_train)


In [ ]:
print(accuracy_score(y_late_train, deep_late.predict(X_train_c)))


In [ ]:
print(accuracy_score(y_late_test, deep_late.predict(X_test_c)))


In [ ]:
print(precision_score(y_late_test, deep_late.predict(X_test_c), zero_division=0))


In [ ]:
print(recall_score(y_late_test, deep_late.predict(X_test_c), zero_division=0))


In [ ]:
deep_time = DecisionTreeRegressor(max_depth=12, random_state=42)


In [ ]:
deep_time.fit(X_train, y_time_train)


In [ ]:
print(mean_absolute_error(y_time_train, deep_time.predict(X_train)))


In [ ]:
print(mean_absolute_error(y_time_test, deep_time.predict(X_test)))


You should see a classification tree that is almost perfect on the training rows (accuracy of 1.0) and weaker on the test rows than the short tree, especially on recall. The deep regression tree can get a training MAE close to zero, then a test MAE worse than the depth-4 tree.

That gap is the picture of overfitting. When you try different depths in your project, always print train and test metrics together, and prefer the model that holds up on the test set.


**INSTRUCTOR NOTE.** Classifier depth 12: train accuracy 1.0, test 0.81, precision 0.615, recall 0.64, matrix [[65, 10], [9, 16]] if they print it. Regressor depth 12: train MAE 0.06, test MAE 6.89 (depth 4 was 6.16 on the test set). The deeper regression tree is worse on new rows, not better. Underline that.


## 6. Challenge (10 to 15 minutes)

You already have `deep_late`, a classification tree with `max_depth=12`. If you need to, fit it again.

1. Print training accuracy, test accuracy, test precision, test recall, and the test confusion matrix.
2. Write three sentences: is the deeper tree better than the depth-4 tree on the test set? What does the train-versus-test gap tell you? Which model would you actually take to GreenLine (dummy, logistic, depth 4, or depth 12), and why?


In [ ]:
# SOLUTION
print(accuracy_score(y_late_train, deep_late.predict(X_train_c)))
print(accuracy_score(y_late_test, deep_late.predict(X_test_c)))
print(precision_score(y_late_test, deep_late.predict(X_test_c), zero_division=0))
print(recall_score(y_late_test, deep_late.predict(X_test_c), zero_division=0))
print(confusion_matrix(y_late_test, deep_late.predict(X_test_c)))


**SOLUTION — sentences.** No: test accuracy is about 0.81 (against 0.82 at depth 4) and recall falls to about 0.64 (against 0.72). The training accuracy of 1.0 with a weaker test score is the sign that the tree has memorised the training rows. The model to take to GreenLine on this evidence is logistic regression: it has the strongest test accuracy and recall, and it beats both the dummy and the trees here.

**INSTRUCTOR NOTE. Challenge.** Students may argue for the depth-4 tree if they prefer a model they can draw. That is a fair communication point, as long as they admit it loses on the test metrics. Do not accept the depth-12 tree on the basis of the training score.


## Recap

- A Decision Tree is a stack of yes or no questions. `max_depth` is a choice, not a quality score.
- For classes, read accuracy *with* precision, recall and the confusion matrix. A dummy can look accurate and still be useless.
- For numbers, read MAE, RMSE and R-squared against a baseline and against the other model.
- Always report training and test performance when you change depth. A perfect training score with a weaker test score is overfitting.
- The "more advanced" model does not automatically win. On this courier data, logistic regression beat the trees at catching late drops, and linear regression beat the trees at estimating times.

Finish the Introductory ML Project next, and keep the capstone moving in parallel. Week 8 is a studio and presentation week: no new methods, just a finished, honest piece of work.


## Assignment (finish the Introductory ML Project)

**Due:** before our Week 8 Session 1 class. **Submit:** a notebook that runs from top to bottom, through the usual submission form.

Follow `Week7_Introductory_ML_Project.txt`. You now have everything in scope: the workflow, a baseline, Linear or Logistic Regression, a Decision Tree, and the metrics above.

Your notebook should include a clear train/test split, a baseline before any model, at least one linear or logistic model and a Decision Tree, metric interpretation in plain language, and a short limitations / next steps note. Please also add a one-paragraph capstone progress update.

**Rubric (10 percent of the course):** Technical correctness, Problem solving, Analytical reasoning, Interpretation, Reproducibility, Documentation. Missing baseline, skipped split, or obvious leakage will be penalised. Accuracy-only on an imbalanced class problem will also be penalised.

AI tools are allowed, but you must be able to explain every line and every metric on our call.
